In [5]:
from pydantic_ai import Agent
from pydantic_ai.models.openrouter import OpenRouterModelSettings
from pydantic_ai.providers.openrouter import OpenRouterProvider
from pydantic import BaseModel, Field
from enum import Enum

settings = OpenRouterModelSettings(
    temperature=0.0,
    max_tokens=10000,
    top_p=1.0, #
    frequency_penalty=0.0, #（频率惩罚）
    presence_penalty=0.0, #（存在惩罚）
    openrouter_provider={"require_parameters": True},
)
model = OpenRouterProvider.model_profile("xiaomi/mimo-v2-flash")
user_input = "我需要生产100个产品，请帮我排程。"

instructions = """
你是APS生产排程系统的主控Agent。

你的职责是：
1. 理解用户的生产排程请求
2. 分析用户的意图和优先级偏好
3. 确定需要执行的子任务序列

分析用户输入时，请注意：
- "交期"、"准时"、"不延误" → 优先准时交付策略
- "换产"、"清洗"、"切换" → 最小化换产策略
- "利润"、"收益" → 最大化利润策略
- 如果没有明确偏好 → 使用均衡策略

返回结构化的响应，包含：
- user_intent: 用户意图的简明描述
- required_tasks: 需要执行的任务（通常为[plan, schedule, explain]）
- strategy_hint: 策略提示
"""
class TaskType(str, Enum):
    PLAN = "plan"
    SCHEDULE = "schedule"
    VALIDATE = "validate"
    ADJUST = "adjust"
    EXPLAIN = "explain"
    MONITOR = "monitor"

class OrchestratorResponse(BaseModel):
    """主控Agent响应"""
    user_intent: str = Field(..., description="用户意图分析")
    required_tasks: list[TaskType] = Field(
        default_factory=lambda: [TaskType.PLAN, TaskType.SCHEDULE, TaskType.EXPLAIN],
        description="需要执行的任务序列",
    )
    strategy_hint: str | None = Field(
        default=None, description="策略提示（如'优先交期'、'最小换产'）"
    )

class AgentIndustry:
    """Agent 工厂：保存参数，调用 build() 时再创建 pydantic_ai.Agent。"""

    def __init__(
        self,
        settings: OpenRouterModelSettings,
        instructions: str,
        model,
        output_type: type[BaseModel],
    ):
        self.model = model
        self.settings = settings
        self.instructions = instructions
        self.output_type = output_type

    def build(self) -> Agent:
        return Agent(
            self.model,
            model_settings=self.settings,
            instructions=self.instructions,
            output_type=self.output_type,
        )

class OrchestratorAgent:
    """主控 Agent：默认用上面的全局 settings / instructions / model；构造时可覆盖任意一项。"""

    def __init__(
        self,
        *,
        agent_settings: OpenRouterModelSettings | None = None,
        agent_instructions: str | None = None,
        agent_model=None,
        agent_output_type: type[BaseModel] | None = None,
    ):
        self.agent = AgentIndustry(
            settings=agent_settings or settings,
            instructions=agent_instructions or instructions,
            model=agent_model or model,
            output_type=agent_output_type or OrchestratorResponse,
        ).build()

    async def run(self, user_input: str) -> BaseModel:
        result = await self.agent.run(user_input)
        return result.output


# 默认：orch = OrchestratorAgent()
# 自定义：OrchestratorAgent(agent_model=OpenRouterProvider.model_profile("其它模型"))

In [3]:
# 内核须能 import aps（如 PYTHONPATH=Z:\pyagent）、OpenRouter 与 Logfire（可选 LOGFIRE_TOKEN）
from aps.core.logfire_setup import init_logfire
from aps.agents.orchestrator import OrchestratorAgent

init_logfire()

async def main():
    agent = OrchestratorAgent()
    text = "本周订单务必准时交付，可以少换几次产"
    out = await agent.run(text)
    print("user_intent:", out.user_intent)
    print("required_tasks:", out.required_tasks)
    print("strategy_hint:", out.strategy_hint)

await main()

Logfire project URL: https://logfire-us.pydantic.dev/evango/starter-project

22:02:39.261 agent run
22:02:39.263   chat xiaomi/mimo-v2-flash
user_intent: 用户希望本周订单准时交付，同时尽量减少换产次数
required_tasks: [<TaskType.PLAN: 'plan'>, <TaskType.SCHEDULE: 'schedule'>, <TaskType.EXPLAIN: 'explain'>]
strategy_hint: 优先交期，最小换产


In [ ]:
# 接续：测试 PlannerAgent（需已能 import aps、OpenRouter；Logfire 可选，与上一格一致可先 init_logfire）
from aps.agents.base import AgentContext
from aps.agents.planner import PlannerAgent

text = "本周订单务必准时交付，可以少换几次产"

ctx = AgentContext(
    orders_info="- O1: 产品A 500件, 截止24h\n- O2: 产品B 300件, 截止12h",
    machines_info="- M1(线1): 支持类型[beverage, dairy]\n- M2(线2): 支持类型[beverage, juice]",
)

async def try_planner():
    planner = PlannerAgent()
    out = await planner.run(text, ctx)
    print("strategy:", out.strategy)
    print("weights:", out.weights.model_dump())
    print("time_limit_seconds:", out.time_limit_seconds)
    print("allow_delays:", out.allow_delays, "max_delay_hours:", out.max_delay_hours)
    print("explanation:", out.explanation)
    params = out.to_optimization_params()
    print("OptimizationParams:", params.model_dump())

await try_planner()
